In [2]:
# Tessellation Discovery Algorithm
# Complete algorithm for exact tessellation discovery in discrete grids.

In [3]:
import json
from collections import deque
from time import perf_counter
from typing import List, Tuple, Dict, Any, Optional, Set

Matrix = List[List[str]]
BLANK = "-"

In [4]:
def shape(M: Matrix) -> Tuple[int, int]:
    return (len(M), len(M[0]) if M else 0)

def non_empty(M: Matrix) -> bool:
    return bool(M) and bool(M[0])

def area(M: Matrix) -> int:
    r, c = shape(M)
    return r * c

def meets_min_size(h: int, w: int) -> bool:
    return (h >= 1 and w >= 2) or (h >= 2 and w >= 1)

def mat_key(M: Matrix) -> Tuple[Tuple[str, ...], ...]:
    return tuple(tuple(r) for r in M)

def equal_mats(A: Matrix, B: Matrix) -> bool:
    return mat_key(A) == mat_key(B)

def is_all_blanks(M: Matrix) -> bool:
    for row in M:
        for cell in row:
            if cell != BLANK:
                return False
    return True

def fmt_mat(M: Matrix, indent: int = 4) -> str:
    pad = " " * indent
    if not M:
        return pad + "[]"
    return "\n".join(pad + " ".join(str(x) for x in row) for row in M)

def is_blank_row(row: List[str]) -> bool:
    return all(x == BLANK for x in row)

def is_blank_col(M: Matrix, j: int) -> bool:
    return all(row[j] == BLANK for row in M)

def trim_outer_blank_border(M: Matrix) -> Matrix:
    R, C = shape(M)
    if R == 0 or C == 0:
        return []
    top = 0
    while top < R and is_blank_row(M[top]):
        top += 1
    bot = R - 1
    while bot >= 0 and is_blank_row(M[bot]):
        bot -= 1
    if top > bot:
        return []
    left = 0
    while left < C and is_blank_col(M, left):
        left += 1
    right = C - 1
    while right >= 0 and is_blank_col(M, right):
        right -= 1
    if left > right:
        return []
    return [row[left:right+1] for row in M[top:bot+1]]

def deduplicate_solutions(solutions: List[List[Matrix]]) -> List[List[Matrix]]:
    seen_signatures = set()
    unique_solutions = []
    for solution in solutions:
        signature = frozenset(mat_key(tile) for tile in solution)
        if signature not in seen_signatures:
            seen_signatures.add(signature)
            unique_solutions.append(solution)
    return unique_solutions

In [5]:
def shave_top(M: Matrix) -> Matrix:
    return M[1:] if len(M) >= 2 else []

def shave_bottom(M: Matrix) -> Matrix:
    return M[:-1] if len(M) >= 2 else []

def shave_left(M: Matrix) -> Matrix:
    if not M or not M[0] or len(M[0]) < 2:
        return []
    return [row[1:] for row in M]

def shave_right(M: Matrix) -> Matrix:
    if not M or not M[0] or len(M[0]) < 2:
        return []
    return [row[:-1] for row in M]

In [6]:
def split_rows_with_duplication(X: Matrix) -> Optional[Tuple[Matrix, Matrix]]:
    R, C = shape(X)
    if R < 2:
        return None
    if R % 2 == 1:
        mid = R // 2
        Xtest = X[:mid+1] + [X[mid]] + X[mid+1:]
        R2 = R + 1
        top = Xtest[:R2//2]
        bot = Xtest[R2//2:]
    else:
        top = X[:R//2]
        bot = X[R//2:]
    return top, bot

def split_cols_with_duplication(X: Matrix) -> Optional[Tuple[Matrix, Matrix]]:
    R, C = shape(X)
    if C < 2:
        return None
    if C % 2 == 1:
        mid = C // 2
        Xtest = [row[:mid+1] + [row[mid]] + row[mid+1:] for row in X]
        C2 = C + 1
        left = [r[:C2//2] for r in Xtest]
        right = [r[C2//2:] for r in Xtest]
    else:
        left = [r[:C//2] for r in X]
        right = [r[C//2:] for r in X]
    return left, right

def split_rows_no_duplication(X: Matrix) -> Optional[Tuple[Matrix, Matrix]]:
    R, C = shape(X)
    if R < 2 or R % 2 != 0:
        return None
    return X[:R//2], X[R//2:]

def split_cols_no_duplication(X: Matrix) -> Optional[Tuple[Matrix, Matrix]]:
    R, C = shape(X)
    if C < 2 or C % 2 != 0:
        return None
    return [r[:C//2] for r in X], [r[C//2:] for r in X]

In [7]:
def tiles_matrix(T: Matrix, X: Matrix) -> bool:
    Th, Tw = shape(T)
    Xh, Xw = shape(X)
    if Th == 0 or Tw == 0 or Xh == 0 or Xw == 0:
        return False
    if Xh % Th != 0 or Xw % Tw != 0:
        return False
    for i in range(Xh):
        for j in range(Xw):
            if X[i][j] != T[i % Th][j % Tw]:
                return False
    return True

def try_inspection(grid: Matrix, label: str) -> Optional[Matrix]:
    print(f"\n{label}:")
    hr = split_rows_with_duplication(grid)
    if hr is not None:
        top, bot = hr
        if equal_mats(top, bot) and meets_min_size(*shape(top)) and not is_all_blanks(top):
            print(f"  ✓ Row overlap found!")
            print(f"    Composite: {shape(top)[0]}×{shape(top)[1]}")
            return top
    hc = split_cols_with_duplication(grid)
    if hc is not None:
        left, right = hc
        if equal_mats(left, right) and meets_min_size(*shape(left)) and not is_all_blanks(left):
            print(f"  ✓ Column overlap found!")
            print(f"    Composite: {shape(left)[0]}×{shape(left)[1]}")
            return left
    print(f"  ✗ No overlap found")
    return None

def check_overlap(node: Matrix) -> Optional[Tuple[Matrix, str]]:
    sr = split_rows_no_duplication(node)
    if sr is not None:
        top, bot = sr
        if equal_mats(top, bot) and meets_min_size(*shape(top)) and not is_all_blanks(top):
            return top, "row"
    sc = split_cols_no_duplication(node)
    if sc is not None:
        left, right = sc
        if equal_mats(left, right) and meets_min_size(*shape(left)) and not is_all_blanks(left):
            return left, "col"
    return None

In [8]:
def bfs_prune_composites(root: Matrix) -> Tuple[List[Matrix], List[List[Matrix]]]:
    Q = deque([root])
    seen: Set[Tuple[Tuple[str, ...], ...]] = {mat_key(root)}
    depth: Dict[Tuple[Tuple[str, ...], ...], int] = {mat_key(root): 0}
    tiles_by_level: Dict[int, List[Matrix]] = {0: [root]}
    composites = []

    while Q:
        node = Q.popleft()
        d = depth[mat_key(node)]
        h, w = shape(node)
        if not meets_min_size(h, w):
            continue
        ov = check_overlap(node)
        if ov is not None:
            seed, kind = ov
            composites.append(seed)
            continue
        for child in [shave_top(node), shave_bottom(node), shave_left(node), shave_right(node)]:
            if not non_empty(child):
                continue
            ch, cw = shape(child)
            if not meets_min_size(ch, cw):
                continue
            k = mat_key(child)
            if k not in seen:
                seen.add(k)
                Q.append(child)
                depth[k] = d + 1
                next_level = d + 1
                if next_level not in tiles_by_level:
                    tiles_by_level[next_level] = []
                tiles_by_level[next_level].append(child)

    max_level = max(tiles_by_level.keys()) if tiles_by_level else 0
    levels = [tiles_by_level.get(i, []) for i in range(max_level + 1)]
    uniq: Dict[Tuple[Tuple[str, ...], ...], Matrix] = {}
    for c in composites:
        if not is_all_blanks(c):
            k = mat_key(c)
            if k not in uniq:
                uniq[k] = c
    return list(uniq.values()), levels

In [9]:
def discover_composites(M: Matrix):
    print(f"\n{'='*80}")
    print("STEP 1: COMPOSITE DISCOVERY")
    print(f"{'='*80}")
    t0 = perf_counter()
    
    composite = try_inspection(M, "Inspection on UNTRIMMED grid")
    if composite is not None:
        dur_ms = (perf_counter() - t0) * 1000.0
        return [composite], dur_ms
    
    trimmed = trim_outer_blank_border(M)
    composite = try_inspection(trimmed, "Inspection on TRIMMED grid")
    if composite is not None:
        dur_ms = (perf_counter() - t0) * 1000.0
        return [composite], dur_ms
    
    print(f"\nBFS Pruning on TRIMMED grid (without duplication):")
    composites, levels = bfs_prune_composites(trimmed)
    dur_ms = (perf_counter() - t0) * 1000.0
    print(f"\nComposites Found: {len(composites)}")
    print(f"Time: {dur_ms:.2f}ms")
    return composites, dur_ms

In [10]:
def normalize_composite(tile: Matrix, comp_id: int):
    print(f"\n{'='*80}")
    print(f"STEP 2: NORMALIZATION - Composite #{comp_id}")
    print(f"{'='*80}")
    t0 = perf_counter()
    Tc = tile
    reductions = 0
    
    while True:
        accepted = False
        sr = split_rows_with_duplication(Tc)
        if sr is not None:
            top, bot = sr
            if equal_mats(top, bot) and meets_min_size(*shape(top)):
                Tc = top
                reductions += 1
                accepted = True
                continue
        sc = split_cols_with_duplication(Tc)
        if sc is not None:
            left, right = sc
            if equal_mats(left, right) and meets_min_size(*shape(left)):
                Tc = left
                reductions += 1
                accepted = True
                continue
        if not accepted:
            break
    
    dur_ms = (perf_counter() - t0) * 1000.0
    print(f"Final: {shape(Tc)[0]}×{shape(Tc)[1]}, Time: {dur_ms:.2f}ms")
    return Tc, dur_ms

In [11]:
def find_tessellations(Tc: Matrix):
    tessellated = []
    h, w = shape(Tc)
    for tile_w in range(1, w):
        if w % tile_w != 0:
            continue
        tile = [row[:tile_w] for row in Tc]
        if tiles_matrix(tile, Tc):
            reduction_count = w // tile_w
            if reduction_count > 1:
                tessellated.append({"tile": tile, "max_uses": reduction_count})
    for tile_h in range(1, h):
        if h % tile_h != 0:
            continue
        tile = Tc[:tile_h]
        if tiles_matrix(tile, Tc):
            reduction_count = h // tile_h
            if reduction_count > 1:
                if not any(equal_mats(t["tile"], tile) for t in tessellated):
                    tessellated.append({"tile": tile, "max_uses": reduction_count})
    return tessellated

In [12]:
def bfs_prune_primes(root: Matrix) -> Tuple[List[List[Matrix]], Dict]:
    if not non_empty(root):
        return [], {}
    Q = deque([root])
    seen: Set[Tuple[Tuple[str, ...], ...]] = {mat_key(root)}
    depth_map: Dict[Tuple[Tuple[str, ...], ...], int] = {mat_key(root): 0}
    tiles_by_level: Dict[int, List[Matrix]] = {0: [root]}
    tessellations_found = {}

    while Q:
        node = Q.popleft()
        d = depth_map[mat_key(node)]
        ov = check_overlap(node)
        if ov is not None:
            seed, kind = ov
            normalized = seed
            while True:
                reduced = False
                sr = split_rows_with_duplication(normalized)
                if sr is not None:
                    top, bot = sr
                    if equal_mats(top, bot) and meets_min_size(*shape(top)):
                        normalized = top
                        reduced = True
                        continue
                sc = split_cols_with_duplication(normalized)
                if sc is not None:
                    left, right = sc
                    if equal_mats(left, right) and meets_min_size(*shape(left)):
                        normalized = left
                        reduced = True
                        continue
                if not reduced:
                    break
            if tiles_matrix(normalized, node):
                h, w = shape(node)
                nh, nw = shape(normalized)
                max_uses = max(h // nh, w // nw)
                if max_uses > 1:
                    tessellations_found[mat_key(normalized)] = {"tile": normalized, "max_uses": max_uses}
            next_level = d
            if next_level not in tiles_by_level:
                tiles_by_level[next_level] = []
            k_normalized = mat_key(normalized)
            if k_normalized not in seen:
                tiles_by_level[next_level].append(normalized)
                seen.add(k_normalized)
            continue

        children = [shave_top(node), shave_bottom(node), shave_left(node), shave_right(node)]
        for child in children:
            if not non_empty(child):
                continue
            h, w = shape(child)
            if not meets_min_size(h, w):
                continue
            if is_all_blanks(child):
                continue
            k = mat_key(child)
            if k not in seen:
                seen.add(k)
                Q.append(child)
                depth_map[k] = d + 1
                next_level = d + 1
                if next_level not in tiles_by_level:
                    tiles_by_level[next_level] = []
                tiles_by_level[next_level].append(child)

    max_level = max(tiles_by_level.keys()) if tiles_by_level else 0
    levels = [tiles_by_level.get(i, []) for i in range(max_level + 1)]
    return levels, tessellations_found

In [13]:
def solve_puzzle(Tc: Matrix, tiles: List[Matrix], tess_info: Dict) -> Tuple[List[List[Matrix]], int]:
    h, w = shape(Tc)
    covered = [[False] * w for _ in range(h)]
    solutions = []
    best_steps = [float('inf')]

    def calc_steps(usage):
        return sum(usage.values())

    def can_place(tile, row, col):
        th, tw = shape(tile)
        if row + th > h or col + tw > w:
            return False
        for i in range(th):
            for j in range(tw):
                if covered[row + i][col + j]:
                    return False
                if Tc[row + i][col + j] != tile[i][j]:
                    return False
        return True

    def place(tile, row, col):
        th, tw = shape(tile)
        for i in range(th):
            for j in range(tw):
                covered[row + i][col + j] = True

    def unplace(tile, row, col):
        th, tw = shape(tile)
        for i in range(th):
            for j in range(tw):
                covered[row + i][col + j] = False

    def is_complete():
        return all(covered[i][j] for i in range(h) for j in range(w))

    def find_next():
        for i in range(h):
            for j in range(w):
                if not covered[i][j]:
                    return (i, j)
        return None

    def backtrack(used, usage):
        if is_complete():
            steps = calc_steps(usage)
            unique = []
            seen = set()
            for tile in used:
                k = mat_key(tile)
                if k not in seen:
                    seen.add(k)
                    unique.append(tile)
            if steps < best_steps[0]:
                best_steps[0] = steps
                solutions.clear()
                solutions.append(unique)
            elif steps == best_steps[0]:
                solutions.append(unique)
            return
        if calc_steps(usage) >= best_steps[0]:
            return
        pos = find_next()
        if pos is None:
            return
        row, col = pos
        for tile in tiles:
            k = mat_key(tile)
            curr = usage.get(k, 0)
            if k in tess_info:
                if curr >= tess_info[k]["max_uses"]:
                    continue
            else:
                if curr >= 1:
                    continue
            if can_place(tile, row, col):
                place(tile, row, col)
                usage[k] = curr + 1
                backtrack(used + [tile], usage)
                unplace(tile, row, col)
                usage[k] = curr

    backtrack([], {})
    return solutions, best_steps[0] if solutions else 0

In [14]:
def extract_primes_hierarchical(normalized_composites: List[Tuple[int, Matrix]]):
    print(f"\n{'='*80}")
    print("STEP 3: HIERARCHICAL PRIME EXTRACTION")
    print(f"{'='*80}")
    t0 = perf_counter()
    sorted_composites = sorted(normalized_composites, key=lambda x: area(x[1]), reverse=True)
    already_handled = set()
    all_primes: Dict[int, List[Matrix]] = {}
    composite_status: Dict[int, str] = {}
    cumulative_results: Dict[int, Tuple[List[List[Matrix]], int]] = {}
    per_level_results: Dict[int, Dict[int, Tuple[List[List[Matrix]], int]]] = {}

    for comp_id, Tc in sorted_composites:
        print(f"\nProcessing Composite #{comp_id}: {shape(Tc)[0]}×{shape(Tc)[1]}")
        k = mat_key(Tc)
        if k in already_handled:
            print(f"  ⊗ SKIPPED")
            composite_status[comp_id] = "SKIPPED"
            continue
        composite_status[comp_id] = "PROCESSED"
        
        tess_original = find_tessellations(Tc)
        levels, tess_from_bfs = bfs_prune_primes(Tc)
        tess_info = {}
        for t in tess_original:
            tess_info[mat_key(t["tile"])] = t
        for k_tile, t in tess_from_bfs.items():
            if k_tile not in tess_info:
                tess_info[k_tile] = t
        
        Tc_key = mat_key(Tc)
        levels_compatible = []
        for i, level_tiles in enumerate(levels):
            compatible = []
            seen = set()
            for tile in level_tiles:
                k_tile = mat_key(tile)
                if not is_all_blanks(tile) and k_tile != Tc_key:
                    if k_tile not in seen:
                        seen.add(k_tile)
                        compatible.append(tile)
            levels_compatible.append(compatible)
        
        tess_tiles = [t["tile"] for t in tess_info.values()]
        all_bfs_primes = []
        seen_primes = set()
        for i in range(1, len(levels_compatible)):
            for tile in levels_compatible[i]:
                k_tile = mat_key(tile)
                if k_tile not in seen_primes:
                    seen_primes.add(k_tile)
                    all_bfs_primes.append(tile)
        
        all_primes_for_composite = tess_tiles + all_bfs_primes
        unique_primes = []
        seen_final = set()
        for p in all_primes_for_composite:
            k_p = mat_key(p)
            if k_p not in seen_final:
                seen_final.add(k_p)
                unique_primes.append(p)
        
        all_primes[comp_id] = unique_primes
        for prime in unique_primes:
            already_handled.add(mat_key(prime))
        
        # Cumulative strategy
        all_cumulative_solutions = []
        best_cumulative_steps = float('inf')
        for end_level in range(1, len(levels_compatible)):
            tiles = []
            for i in range(1, end_level + 1):
                tiles.extend(levels_compatible[i])
            all_tiles = tess_tiles + tiles
            seen = set()
            unique = []
            for t in all_tiles:
                k_t = mat_key(t)
                if k_t not in seen:
                    seen.add(k_t)
                    unique.append(t)
            unique.sort(key=area, reverse=True)
            solutions, steps = solve_puzzle(Tc, unique, tess_info)
            if steps < best_cumulative_steps:
                best_cumulative_steps = steps
                all_cumulative_solutions = solutions
            elif steps == best_cumulative_steps:
                all_cumulative_solutions.extend(solutions)
        
        best_cumulative_solutions = deduplicate_solutions(all_cumulative_solutions)
        cumulative_results[comp_id] = (best_cumulative_solutions, best_cumulative_steps)
        
        # Per-level strategy
        per_level_for_composite = {}
        for level_idx in range(1, len(levels_compatible)):
            if not levels_compatible[level_idx]:
                continue
            tiles = tess_tiles + levels_compatible[level_idx]
            seen = set()
            unique = []
            for t in tiles:
                k_t = mat_key(t)
                if k_t not in seen:
                    seen.add(k_t)
                    unique.append(t)
            unique.sort(key=area, reverse=True)
            solutions, steps = solve_puzzle(Tc, unique, tess_info)
            if solutions:
                unique_solutions = deduplicate_solutions(solutions)
                per_level_for_composite[level_idx] = (unique_solutions, steps)
        
        per_level_results[comp_id] = per_level_for_composite

    total_dur = (perf_counter() - t0) * 1000.0
    return all_primes, composite_status, cumulative_results, per_level_results, total_dur

In [18]:
def print_final_summary(normalized_composites: List[Tuple[int, Matrix]],
                       all_primes: Dict[int, List[Matrix]],
                       composite_status: Dict[int, str],
                       cumulative_results: Dict[int, Tuple[List[List[Matrix]], int]],
                       per_level_results: Dict[int, Dict[int, Tuple[List[List[Matrix]], int]]]):
    print(f"\n{'█'*80}")
    print(f"{'█'*80}")
    print(f"FINAL SUMMARY")
    print(f"{'█'*80}")
    print(f"{'█'*80}")

    # COMPOSITES
    print(f"\n{'='*80}")
    print(f"ALL COMPOSITES (Normalized)")
    print(f"{'='*80}")

    for comp_id, Tc in normalized_composites:
        status = composite_status.get(comp_id, "UNKNOWN")
        status_mark = "✓" if status == "PROCESSED" else "⊗"
        print(f"\n{status_mark} Composite #{comp_id}: {shape(Tc)[0]}×{shape(Tc)[1]} [{status}]")
        print(fmt_mat(Tc))

    # PRIMES
    print(f"\n{'='*80}")
    print(f"ALL PRIMES (By Composite)")
    print(f"{'='*80}")

    for comp_id, Tc in normalized_composites:
        if comp_id in all_primes:
            primes = all_primes[comp_id]
            print(f"\nComposite #{comp_id} → {len(primes)} Prime(s):")
            for i, prime in enumerate(primes, 1):
                print(f"\n  Prime #{i}: {shape(prime)[0]}×{shape(prime)[1]}")
                print(fmt_mat(prime, indent=6))
        else:
            print(f"\nComposite #{comp_id} → SKIPPED")

    # OPTIMAL SOLUTIONS - CUMULATIVE
    print(f"\n{'='*80}")
    print(f"OPTIMAL SOLUTIONS - CUMULATIVE STRATEGY")
    print(f"{'='*80}")

    for comp_id, Tc in normalized_composites:
        if comp_id in cumulative_results:
            solutions, steps = cumulative_results[comp_id]
            print(f"\nComposite #{comp_id}: {len(solutions)} unique solution(s), {steps} steps")

            for sol_idx, solution in enumerate(solutions, 1):
                print(f"\n  Solution #{sol_idx}: {len(solution)} unique tile(s)")
                for tile_idx, tile in enumerate(solution, 1):
                    print(f"\n    Tile #{tile_idx}: {shape(tile)[0]}×{shape(tile)[1]}")
                    print(fmt_mat(tile, indent=8))

    # OPTIMAL SOLUTIONS - PER-LEVEL
    print(f"\n{'='*80}")
    print(f"OPTIMAL SOLUTIONS - PER-LEVEL STRATEGY")
    print(f"{'='*80}")

    for comp_id, Tc in normalized_composites:
        if comp_id in per_level_results:
            level_sols = per_level_results[comp_id]
            print(f"\nComposite #{comp_id}:")

            for level_idx in sorted(level_sols.keys()):
                solutions, steps = level_sols[level_idx]
                print(f"\n  Level {level_idx}: {len(solutions)} unique solution(s), {steps} steps")

                for sol_idx, solution in enumerate(solutions, 1):
                    print(f"\n    Solution #{sol_idx}: {len(solution)} unique tile(s)")
                    for tile_idx, tile in enumerate(solution, 1):
                        print(f"\n      Tile #{tile_idx}: {shape(tile)[0]}×{shape(tile)[1]}")
                        print(fmt_mat(tile, indent=10))

    # STATISTICS
    print(f"\n{'='*80}")
    print(f"STATISTICS")
    print(f"{'='*80}")

    total_composites = len(normalized_composites)
    processed = sum(1 for s in composite_status.values() if s == "PROCESSED")
    skipped = sum(1 for s in composite_status.values() if s == "SKIPPED")
    total_primes = sum(len(primes) for primes in all_primes.values())

    print(f"  Total Composites:      {total_composites}")
    print(f"  Processed:             {processed}")
    print(f"  Skipped (redundant):   {skipped}")
    print(f"  Total Primes Found:    {total_primes}")
    print(f"{'='*80}\n")


# Load and run
with open('../data/data.json', 'r') as f:
    data = json.load(f)

# Run each test case
for test_data in data['test_cases']:
    print(f"\n{'█'*80}")
    print(f"CASE: {test_data['name']}")
    print(f"{'█'*80}")
    
    grid = test_data['grid']

# Pipeline
composites, comp_time = discover_composites(grid)

normalized = []
norm_time = 0
for i, comp in enumerate(composites, 1):
    Tc, dur = normalize_composite(comp, i)
    normalized.append((i, Tc))
    norm_time += dur

all_primes, composite_status, cumulative_results, per_level_results, prime_time = \
    extract_primes_hierarchical(normalized)

# Print full summary
print_final_summary(normalized, all_primes, composite_status, cumulative_results, per_level_results)

# Timing
total_time = comp_time + norm_time + prime_time
print(f"\n{'='*80}")
print(f"TIMING SUMMARY")
print(f"{'='*80}")
print(f"  Composite Discovery:  {comp_time:>10.2f}ms")
print(f"  Normalization:        {norm_time:>10.2f}ms")
print(f"  Prime Extraction:     {prime_time:>10.2f}ms")
print(f"  {'─'*40}")
print(f"  TOTAL:                {total_time:>10.2f}ms")
print(f"{'='*80}\n")


████████████████████████████████████████████████████████████████████████████████
CASE: Mixed-noise
████████████████████████████████████████████████████████████████████████████████

STEP 1: COMPOSITE DISCOVERY

Inspection on UNTRIMMED grid:
  ✗ No overlap found

Inspection on TRIMMED grid:
  ✗ No overlap found

BFS Pruning on TRIMMED grid (without duplication):

Composites Found: 14
Time: 1.64ms

STEP 2: NORMALIZATION - Composite #1
Final: 2×4, Time: 0.01ms

STEP 2: NORMALIZATION - Composite #2
Final: 2×3, Time: 0.01ms

STEP 2: NORMALIZATION - Composite #3
Final: 2×3, Time: 0.00ms

STEP 2: NORMALIZATION - Composite #4
Final: 2×2, Time: 0.00ms

STEP 2: NORMALIZATION - Composite #5
Final: 2×2, Time: 0.00ms

STEP 2: NORMALIZATION - Composite #6
Final: 2×2, Time: 0.00ms

STEP 2: NORMALIZATION - Composite #7
Final: 3×1, Time: 0.00ms

STEP 2: NORMALIZATION - Composite #8
Final: 1×2, Time: 0.00ms

STEP 2: NORMALIZATION - Composite #9
Final: 2×1, Time: 0.00ms

STEP 2: NORMALIZATION - Composite